# Studio di Robustezza al Rumore — MAGIC Gamma Telescope

Questo notebook analizza **quanto i modelli di Machine Learning resistono alla "sporcizia" dei dati**.
Confrontiamo tre modelli — **NN Base**, **NN Pro (regolarizzata)** e **Random Forest** — sotto sei tipi di rumore
(*feature noise, label noise random, label noise borderline, missing values, outlier, missing feature*),
ciascuno a tre livelli di intensità: **10%, 25%, 40%**.

Per ogni modello e livello mostriamo, in stile reporting completo: **curve train vs validation**, **confusion matrix**
e **classification report**; al termine di ogni sezione, **grafici di confronto** dei modelli su *accuracy* e *F1*; e in chiusura un
**confronto aggregato** modello per modello su tutti i rumori, più gli **intervalli di confidenza**.

> ⚠️ **Prerequisito ambiente.** Le reti neurali usano **TensorFlow/Keras**. TensorFlow **non supporta Python 3.14**:
> eseguire questo notebook in un ambiente **Python 3.11 / 3.12 con `tensorflow` installato**
> (`pip install tensorflow scikit-learn pandas matplotlib seaborn scipy`).

# Inizializzazione e Caricamento Dati
Importiamo le librerie e carichiamo il dataset MAGIC Gamma Telescope (10 feature continue + 1 label: h=hadron, g=gamma).

In [ ]:
import os
import math
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    recall_score, f1_score, roc_curve, roc_auc_score, log_loss
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, adjusted_mutual_info_score
)
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats
from scipy.stats.mstats import winsorize

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('Librerie caricate OK — TensorFlow', tf.__version__)

In [ ]:
# Caricamento del dataset
col_names = ['fLength', 'fWidth', 'fSize', 'fConc', 'fConc1',
             'fAsym', 'fM3Long', 'fM3Trans', 'fAlpha', 'fDist', 'class']

df = pd.read_csv('magic04.data', names=col_names)
df['class'] = df['class'].map({'h': 0, 'g': 1})

print('Dataset caricato con successo in locale! Dimensioni:', df.shape)

## Suddivisione del Dataset (Train, Val, Test)
Dividiamo in 60% Training, 20% Validation, 20% Test. Il validation set serve a monitorare l'overfitting
durante l'addestramento (curve train vs validation) e l'early stopping della rete Pro.

In [ ]:
X = df.drop(['class'], axis=1)
y = df['class']
seed = 42

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=seed, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=seed, stratify=y_temp)

df_train = pd.concat([X_train, y_train], axis=1)
print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

# Analisi Esplorativa
Verifichiamo l'assenza di valori nulli, rimuoviamo i duplicati e osserviamo distribuzioni, separabilità delle
classi (boxplot) e correlazioni tra feature.

In [ ]:
print('Valori mancanti per colonna:\n', df.isnull().sum())
print(f'\nDuplicati trovati prima della pulizia: {df_train.duplicated().sum()}')
df_train = df_train.drop_duplicates()
print('Dataset Training dopo rimozione duplicati:', df_train.shape)

In [ ]:
signal_class = df_train['class'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(signal_class, labels=['gamma (1)', 'hadron (0)'],
        autopct='%1.1f%%', startangle=90, colors=['#ff9999', '#66b3ff'])
plt.title('Distribuzione del Target nel Training Set')
plt.axis('equal'); plt.show()

In [ ]:
numeric_cols = [c for c in df_train.select_dtypes(include=['float64', 'int64']).columns if c != 'class']
n_cols = 3
n_rows = math.ceil(len(numeric_cols) / n_cols)
plt.figure(figsize=(15, 4 * n_rows))
for i, col in enumerate(numeric_cols):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.histplot(df_train[col], kde=True, bins=30, edgecolor='black')
    plt.title(f'Distribuzione di {col}'); plt.xlabel(''); plt.ylabel('Frequenza')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(15, 4 * n_rows))
for i, col in enumerate(numeric_cols):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.boxplot(x='class', y=col, hue='class', data=df_train, palette='Set2', legend=False)
    plt.title(f'{col} vs Class')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_train[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Matrice di Correlazione'); plt.show()

# Preprocessing e Scalatura
Standardizziamo (media 0, dev.std 1). Lo scaler è addestrato **solo sul Train** per evitare data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

y_train = y_train.astype('int64')
y_val   = y_val.astype('int64')
y_test  = y_test.astype('int64')

feature_names = list(X.columns)
print('Scaling completato. Feature:', feature_names)

# Funzioni di Noise Injection
Definiamo tutte le funzioni che "sporcano" i dati. Ogni tipo di rumore simula un guasto realistico:

| Tipo | Cosa simula |
|------|-------------|
| **Feature noise** | misurazioni imprecise (rumore gaussiano sulle feature) |
| **Label noise random** | errori di etichettatura casuali |
| **Label noise borderline** | errori sui campioni ambigui (vicini al confine decisionale) |
| **Missing values** | celle mancanti (NaN) → imputazione con media o mediana |
| **Outlier** | valori estremi → si possono lasciare, *winsorizzare* o imputare |
| **Missing feature** | uno o più sensori guasti (feature azzerate) |

In [ ]:
def add_gaussian_noise(X, noise_level=0.1, seed=42):
    rng = np.random.default_rng(seed)
    return X + rng.normal(0.0, noise_level, size=X.shape)


def add_label_noise_random(y, noise_level=0.1, seed=42):
    rng = np.random.default_rng(seed)
    y_noisy = np.array(y).copy()
    n_flips = int(len(y_noisy) * noise_level)
    idx = rng.choice(len(y_noisy), size=n_flips, replace=False)
    y_noisy[idx] = 1 - y_noisy[idx]
    return y_noisy

add_label_noise = add_label_noise_random  # alias retrocompatibile


def add_label_noise_borderline(X_scaled, y, noise_level=0.1, seed=42):
    """Inverte le etichette dei campioni piu' vicini al centroide della classe opposta."""
    y_arr = np.array(y).copy()
    n_flips = int(len(y_arr) * noise_level)
    centroids = {c: X_scaled[y_arr == c].mean(axis=0) for c in [0, 1]}
    distances = np.array([np.linalg.norm(X_scaled[i] - centroids[1 - y_arr[i]])
                          for i in range(len(y_arr))])
    borderline_idx = np.argsort(distances)[:n_flips]
    y_arr[borderline_idx] = 1 - y_arr[borderline_idx]
    return y_arr


def add_missing_values(X, missing_rate=0.1, seed=42):
    rng = np.random.default_rng(seed)
    X_missing = X.astype(float).copy()
    X_missing[rng.random(X.shape) < missing_rate] = np.nan
    return X_missing


def impute_missing(X_train_m, X_test_m, strategy='mean'):
    imp = SimpleImputer(strategy=strategy)
    return imp.fit_transform(X_train_m), imp.transform(X_test_m)


def impute_train_val_test(X_tr, X_val, X_te, strategy='mean'):
    """Un solo imputer fittato sul train, applicato a val e test (no leakage)."""
    imp = SimpleImputer(strategy=strategy)
    X_tr_i = imp.fit_transform(X_tr)
    return X_tr_i, imp.transform(X_val), imp.transform(X_te)


def add_outliers(X, outlier_rate=0.05, magnitude=5.0, seed=42):
    rng = np.random.default_rng(seed)
    X_out = X.copy()
    n_out = int(len(X_out) * outlier_rate)
    idx = rng.choice(len(X_out), size=n_out, replace=False)
    sign = rng.choice([-1, 1], size=(n_out, X_out.shape[1]))
    X_out[idx] = sign * magnitude
    return X_out


def winsorize_features(X, limits=(0.05, 0.05)):
    """Tronca le code (5% per lato) di ogni feature: attenua gli outlier senza eliminarli."""
    return np.column_stack([np.asarray(winsorize(X[:, j], limits=limits))
                            for j in range(X.shape[1])])


def impute_outliers(X, thresh=4.0, strategy='median'):
    """Marca come NaN i valori |z|>thresh e li imputa (mediana)."""
    Xc = X.astype(float).copy()
    Xc[np.abs(Xc) > thresh] = np.nan
    return SimpleImputer(strategy=strategy).fit_transform(Xc)


def delete_feature(X, feature_idx):
    """Azzera una o piu' feature (sensore guasto). feature_idx: int o lista."""
    X_del = X.copy()
    idxs = [feature_idx] if np.isscalar(feature_idx) else list(feature_idx)
    for fi in idxs:
        X_del[:, fi] = 0.0
    return X_del


NOISE_LEVELS = [0.10, 0.25, 0.40]
print('Funzioni di noise definite OK | livelli:', NOISE_LEVELS)

# Modelli
Sezione coesa con la definizione dei tre modelli e degli strumenti di valutazione.

1. **NN Base** — `Dense(32) -> Dense(16) -> sigmoid`, Adam, nessuna regolarizzazione.
2. **NN Pro** — stessa architettura + **Dropout** + **L2** + **class_weight** bilanciato + **EarlyStopping** (restore best).
3. **Random Forest** — ensemble di 200 alberi.

Le due reti restituiscono `(model, history)`: l'oggetto `history` di Keras permette di tracciare le **curve train vs
validation** di accuracy e loss. Il Random Forest non ha epoche, quindi per esso mostriamo **solo** confusion matrix +
classification report.

In [ ]:
def train_base_network(X_tr, y_tr, X_val, y_val, epochs=75, batch_size=32, verbose=0, random_state=42):
    tf.keras.utils.set_random_seed(random_state)
    model = Sequential([
        Input(shape=(X_tr.shape[1],)),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(loss='binary_crossentropy', optimizer=tf.keras.optimizers.Adam(1e-3), metrics=['accuracy'])
    history = model.fit(X_tr, y_tr, epochs=epochs, batch_size=batch_size,
                        validation_data=(X_val, y_val), verbose=verbose)
    return model, history


def train_pro_network(X_tr, y_tr, X_val, y_val, epochs=150, batch_size=32, verbose=0,
                      dropout_rate=0.2, l2_lambda=1e-4, patience=15, random_state=42):
    tf.keras.utils.set_random_seed(random_state)
    model = Sequential([
        Input(shape=(X_tr.shape[1],)),
        Dense(32, activation='relu', kernel_regularizer=l2(l2_lambda)),
        Dropout(dropout_rate),
        Dense(16, activation='relu', kernel_regularizer=l2(l2_lambda)),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(loss='binary_crossentropy', optimizer=tf.keras.optimizers.Adam(1e-3), metrics=['accuracy'])
    classes = np.unique(y_tr)
    weights = compute_class_weight('balanced', classes=classes, y=np.array(y_tr))
    cw = {int(c): w for c, w in zip(classes, weights)}
    es = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)
    history = model.fit(X_tr, y_tr, epochs=epochs, batch_size=batch_size,
                        validation_data=(X_val, y_val), class_weight=cw,
                        callbacks=[es], verbose=verbose)
    return model, history


def train_random_forest(X_tr, y_tr, n_estimators=200, random_state=42):
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    return rf

print('Funzioni di training definite OK')

In [ ]:
def model_proba(model, X):
    """Probabilita' della classe positiva, unificata per Keras e sklearn."""
    if hasattr(model, 'predict_proba'):          # Random Forest (sklearn)
        return model.predict_proba(X)[:, 1]
    return np.asarray(model.predict(X, verbose=0)).ravel()  # Keras


def evaluate_network(model, history, X_test, y_test, model_name='Modello'):
    """Report completo per una rete Keras: curve train/val + confusion matrix + classification report."""
    print('\n' + '=' * 60)
    print(f' RISULTATI: {model_name}')
    print('=' * 60)
    proba = np.asarray(model.predict(X_test, verbose=0)).ravel()
    y_pred = (proba > 0.5).astype('int64')
    print(f'Test loss: {log_loss(y_test, proba):.4f} - Test accuracy: {accuracy_score(y_test, y_pred):.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title(f'{model_name} - Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(loc='lower right'); axes[0].grid(True)
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title(f'{model_name} - Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(loc='upper right'); axes[1].grid(True)
    plt.tight_layout(); plt.show()

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'Confusion Matrix — {model_name}')
    plt.show()

    print(f'AUC: {roc_auc_score(y_test, proba):.4f}')
    print('\nReport di Classificazione:')
    print(classification_report(y_test, y_pred, zero_division=0))
    print('=' * 60)


def evaluate_rf(model, X_test, y_test, model_name='RF'):
    """Report per Random Forest: confusion matrix + classification report (niente curve)."""
    print('\n' + '=' * 60)
    print(f' RISULTATI: {model_name}')
    print('=' * 60)
    proba = model.predict_proba(X_test)[:, 1]
    y_pred = (proba > 0.5).astype('int64')
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'Confusion Matrix — {model_name}')
    plt.show()
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
    print(f'AUC: {roc_auc_score(y_test, proba):.4f}')
    print('\nReport di Classificazione:')
    print(classification_report(y_test, y_pred, zero_division=0))
    print('=' * 60)

print('Funzioni di valutazione definite OK')

In [ ]:
PALETTE = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#ff7f0e', '#8c564b']


def compute_metrics(model, X_test, y_test):
    proba = model_proba(model, X_test)
    y_pred = (proba > 0.5).astype(int)
    return {
        'accuracy':     accuracy_score(y_test, y_pred),
        'recall_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_macro':     f1_score(y_test, y_pred, average='macro', zero_division=0),
        'auc':          roc_auc_score(y_test, proba),
    }


def plot_three_models(df_base, df_pro, df_rf, metric='accuracy', tipo='Feature noise', title=None):
    """Confronto NN base vs NN pro vs RF su una metrica, al variare del livello di rumore."""
    plt.figure(figsize=(10, 5.5))
    for df_, color, label, mark in [
        (df_base, PALETTE[0], 'NN Base', 'o'),
        (df_pro,  PALETTE[1], 'NN Pro',  's'),
        (df_rf,   PALETTE[2], 'RF',      '^'),
    ]:
        d = df_.sort_values('livello')
        plt.plot(d['livello'], d[metric], marker=mark, lw=2.2, color=color, label=label, markersize=8)
    plt.xlabel('Livello di rumore (%)'); plt.ylabel(metric.replace('_', ' ').upper())
    plt.title(title or f'Confronto modelli — {tipo} ({metric})', fontweight='bold')
    plt.grid(alpha=0.3); plt.legend(frameon=True); plt.tight_layout(); plt.show()


def plot_aggregate_per_model(model_name, dfs_by_noise, metric='accuracy'):
    """Una figura per modello: una linea per tipo di rumore (x = livello)."""
    plt.figure(figsize=(10, 6))
    for (noise_name, df_), color in zip(dfs_by_noise.items(), PALETTE):
        d = df_.sort_values('livello')
        plt.plot(d['livello'], d[metric], marker='o', lw=2, color=color, label=noise_name)
    plt.xlabel('Livello di rumore (%)'); plt.ylabel(metric.replace('_', ' ').upper())
    plt.title(f'{model_name} — robustezza per tipo di rumore ({metric})', fontweight='bold')
    plt.grid(alpha=0.3); plt.legend(frameon=True, fontsize=9); plt.tight_layout(); plt.show()


def run_noise_section(noise_name, make_data, levels=NOISE_LEVELS, evaluate=True, seed=42):
    """Driver: addestra NN base, NN pro, RF su baseline + livelli, emette i report completi
    e i grafici di confronto, e restituisce {'NN base': df, 'NN pro': df, 'RF': df}.

    make_data(lvl, seed) -> (X_tr, y_tr, X_val, y_val, X_te, y_te)
    """
    rows = {'NN base': [], 'NN pro': [], 'RF': []}
    for lvl in [0.0] + list(levels):
        Xtr, ytr, Xval, yval, Xte, yte = make_data(lvl, seed)
        livello = int(round(lvl * 100))
        key = 'Baseline' if lvl == 0 else f'Rumore {livello}%'

        mb, hb = train_base_network(Xtr, ytr, Xval, yval, random_state=seed)
        mp, hp = train_pro_network(Xtr, ytr, Xval, yval, random_state=seed)
        mr = train_random_forest(Xtr, ytr, random_state=seed)

        if evaluate:
            evaluate_network(mb, hb, Xte, yte, f'NN base — {noise_name} {key}')
            evaluate_network(mp, hp, Xte, yte, f'NN pro — {noise_name} {key}')
            evaluate_rf(mr, Xte, yte, f'RF — {noise_name} {key}')

        for name, model in [('NN base', mb), ('NN pro', mp), ('RF', mr)]:
            rows[name].append({'modello': name, 'tipo_rumore': noise_name,
                               'livello': livello, **compute_metrics(model, Xte, yte)})

    dfs = {m: pd.DataFrame(rows[m]).sort_values('livello').reset_index(drop=True) for m in rows}
    print(f'\n>>> Confronto modelli — {noise_name}')
    plot_three_models(dfs['NN base'], dfs['NN pro'], dfs['RF'], 'accuracy', noise_name)
    plot_three_models(dfs['NN base'], dfs['NN pro'], dfs['RF'], 'f1_macro', noise_name)
    return dfs

print('Driver e funzioni di plotting definiti OK')

## Baseline su dati puliti
Addestriamo una volta i tre modelli sui dati puliti: è il riferimento (livello 0%) per tutte le sezioni.
Calcoliamo anche la **feature importance** del Random Forest, che useremo nella sezione *Missing feature*.

In [ ]:
print('>>> BASELINE su dati puliti')
m_base_clean, h_base_clean = train_base_network(X_train_scaled, y_train, X_val_scaled, y_val, random_state=seed)
evaluate_network(m_base_clean, h_base_clean, X_test_scaled, y_test, 'NN base — Baseline pulita')

m_pro_clean, h_pro_clean = train_pro_network(X_train_scaled, y_train, X_val_scaled, y_val, random_state=seed)
evaluate_network(m_pro_clean, h_pro_clean, X_test_scaled, y_test, 'NN pro — Baseline pulita')

rf_clean = train_random_forest(X_train_scaled, y_train, random_state=seed)
evaluate_rf(rf_clean, X_test_scaled, y_test, 'RF — Baseline pulita')

fi = pd.Series(rf_clean.feature_importances_, index=feature_names).sort_values(ascending=False)
plt.figure(figsize=(9, 5))
sns.barplot(x=fi.values, y=fi.index, palette='viridis')
plt.title('RF — Feature Importance (dati puliti)', fontweight='bold')
plt.xlabel('Importanza media'); plt.tight_layout(); plt.show()

# Dizionario che raccoglie il df canonico di ogni rumore, per il confronto aggregato finale
AGG = {'NN base': {}, 'NN pro': {}, 'RF': {}}

# Sezione A — Feature Noise (rumore gaussiano)
Aggiungiamo rumore gaussiano \(N(0, \sigma)\) alle feature del **training** (val e test restano puliti):
simula sensori imprecisi. Ci aspettiamo un degrado graduale; la rete Pro e il RF dovrebbero resistere meglio.

In [ ]:
def make_fn(lvl, s):
    Xtr = X_train_scaled if lvl == 0 else add_gaussian_noise(X_train_scaled, noise_level=lvl, seed=s)
    return Xtr, np.array(y_train), X_val_scaled, np.array(y_val), X_test_scaled, np.array(y_test)

dfs_fn = run_noise_section('Feature noise', make_fn, seed=seed)
for m in AGG:
    AGG[m]['Feature noise'] = dfs_fn[m]

# Sezione B — Label Noise Random
Invertiamo casualmente una percentuale di etichette del training: errori di annotazione uniformi.
Atteso: le reti tendono a memorizzare gli errori (overfitting visibile nelle curve train vs val); il RF è in genere più robusto.

In [ ]:
def make_ln(lvl, s):
    ytr = np.array(y_train) if lvl == 0 else add_label_noise_random(y_train, noise_level=lvl, seed=s)
    return X_train_scaled, ytr, X_val_scaled, np.array(y_val), X_test_scaled, np.array(y_test)

dfs_ln = run_noise_section('Label noise random', make_ln, seed=seed)
for m in AGG:
    AGG[m]['Label noise random'] = dfs_ln[m]

# Sezione C — Label Noise Borderline
Invertiamo le etichette dei campioni **più ambigui** (vicini al centroide della classe opposta). È un rumore più
"insidioso" del random, perché corrompe proprio gli esempi vicini al confine decisionale.

In [ ]:
def make_lnb(lvl, s):
    ytr = np.array(y_train) if lvl == 0 else add_label_noise_borderline(X_train_scaled, y_train, noise_level=lvl, seed=s)
    return X_train_scaled, ytr, X_val_scaled, np.array(y_val), X_test_scaled, np.array(y_test)

dfs_lnb = run_noise_section('Label noise borderline', make_lnb, seed=seed)
for m in AGG:
    AGG[m]['Label noise borderline'] = dfs_lnb[m]

# Sezione D — Missing Values
Inseriamo NaN in una frazione di celle. Keras e RF non accettano NaN: serve **imputazione**. Proviamo **due strategie**:
imputazione con **media** e con **mediana** (l'imputer è fittato solo sul train). Produciamo il report completo per
entrambe e poi le confrontiamo direttamente. Per il confronto aggregato finale usiamo la **mediana** come variante canonica.

In [ ]:
def make_missing(strategy):
    def _f(lvl, s):
        if lvl == 0:
            return X_train_scaled, np.array(y_train), X_val_scaled, np.array(y_val), X_test_scaled, np.array(y_test)
        Xtr_m  = add_missing_values(X_train_scaled, missing_rate=lvl, seed=s)
        Xval_m = add_missing_values(X_val_scaled,   missing_rate=lvl, seed=s + 1)
        Xte_m  = add_missing_values(X_test_scaled,  missing_rate=lvl, seed=s + 2)
        Xtr_i, Xval_i, Xte_i = impute_train_val_test(Xtr_m, Xval_m, Xte_m, strategy=strategy)
        return Xtr_i, np.array(y_train), Xval_i, np.array(y_val), Xte_i, np.array(y_test)
    return _f

print('### Missing Values — imputazione MEDIA ###')
dfs_miss_mean   = run_noise_section('Missing values (media)',   make_missing('mean'),   seed=seed)
print('### Missing Values — imputazione MEDIANA ###')
dfs_miss_median = run_noise_section('Missing values (mediana)', make_missing('median'), seed=seed)

for m in AGG:
    AGG[m]['Missing values'] = dfs_miss_median[m]  # variante canonica = mediana

In [ ]:
# Confronto diretto media vs mediana (accuracy) per ogni modello
for m in ['NN base', 'NN pro', 'RF']:
    plt.figure(figsize=(8, 5))
    plt.plot(dfs_miss_mean[m]['livello'],   dfs_miss_mean[m]['accuracy'],   marker='o', lw=2, label='Imputazione media')
    plt.plot(dfs_miss_median[m]['livello'], dfs_miss_median[m]['accuracy'], marker='s', lw=2, label='Imputazione mediana')
    plt.xlabel('Livello di rumore (%)'); plt.ylabel('Accuracy')
    plt.title(f'Missing values: media vs mediana — {m}', fontweight='bold')
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

# Sezione E — Outlier
Sostituiamo una frazione di campioni del training con valori estremi (\(\pm 5\,\sigma\)). Confrontiamo **tre strategie**:
**lasciati** (nessuna correzione), **winsorizzazione** (troncamento delle code) e **imputazione** (i valori |z|>4 diventano
NaN e si imputano con la mediana). Report completo per ciascuna; variante canonica per l'aggregato = **winsorizzazione**.

In [ ]:
def make_outlier(strategy):
    def _f(lvl, s):
        if lvl == 0:
            return X_train_scaled, np.array(y_train), X_val_scaled, np.array(y_val), X_test_scaled, np.array(y_test)
        Xtr = add_outliers(X_train_scaled, outlier_rate=lvl, magnitude=5.0, seed=s)
        if strategy == 'winsorize':
            Xtr = winsorize_features(Xtr)
        elif strategy == 'impute':
            Xtr = impute_outliers(Xtr)
        # 'leave' -> nessuna correzione
        return Xtr, np.array(y_train), X_val_scaled, np.array(y_val), X_test_scaled, np.array(y_test)
    return _f

print('### Outlier — LASCIATI ###')
dfs_out_leave = run_noise_section('Outlier (lasciati)',        make_outlier('leave'),     seed=seed)
print('### Outlier — WINSORIZZAZIONE ###')
dfs_out_wins  = run_noise_section('Outlier (winsorizzazione)', make_outlier('winsorize'), seed=seed)
print('### Outlier — IMPUTAZIONE ###')
dfs_out_imp   = run_noise_section('Outlier (imputazione)',     make_outlier('impute'),    seed=seed)

for m in AGG:
    AGG[m]['Outlier'] = dfs_out_wins[m]  # variante canonica = winsorizzazione

In [ ]:
# Confronto diretto delle tre strategie sugli outlier (accuracy) per modello
for m in ['NN base', 'NN pro', 'RF']:
    plt.figure(figsize=(8, 5))
    plt.plot(dfs_out_leave[m]['livello'], dfs_out_leave[m]['accuracy'], marker='o', lw=2, label='Lasciati')
    plt.plot(dfs_out_wins[m]['livello'],  dfs_out_wins[m]['accuracy'],  marker='s', lw=2, label='Winsorizzazione')
    plt.plot(dfs_out_imp[m]['livello'],   dfs_out_imp[m]['accuracy'],   marker='^', lw=2, label='Imputazione')
    plt.xlabel('Livello di rumore (%)'); plt.ylabel('Accuracy')
    plt.title(f'Outlier: strategie a confronto — {m}', fontweight='bold')
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

# Sezione F — Missing Feature (sensore guasto)
Azzeriamo intere feature, simulando sensori guasti. Non esiste un "livello %" continuo: mappiamo i tre livelli sulla
**rimozione delle feature più importanti** (per importanza RF): 10% → top-1, 25% → top-3, 40% → top-5.
La rimozione è applicata in modo coerente a train, validation e test.

In [ ]:
top_feats = list(fi.index)                  # feature ordinate per importanza (desc)
LEVEL_TO_K = {0: 0, 10: 1, 25: 3, 40: 5}    # livello% -> n. feature rimosse

def make_missfeat(lvl, s):
    k = LEVEL_TO_K[int(round(lvl * 100))]
    idxs = [feature_names.index(f) for f in top_feats[:k]]
    Xtr  = delete_feature(X_train_scaled, idxs) if idxs else X_train_scaled
    Xval = delete_feature(X_val_scaled,   idxs) if idxs else X_val_scaled
    Xte  = delete_feature(X_test_scaled,  idxs) if idxs else X_test_scaled
    print(f'  livello {int(round(lvl*100))}% -> rimosse {k} feature: {top_feats[:k]}')
    return Xtr, np.array(y_train), Xval, np.array(y_val), Xte, np.array(y_test)

dfs_mf = run_noise_section('Missing feature', make_missfeat, seed=seed)
for m in AGG:
    AGG[m]['Missing feature'] = dfs_mf[m]

# Confronto Aggregato Finale (modello per modello)
Per ogni modello tracciamo, in un'unica figura, le **6 curve** dei diversi rumori (per missing values e outlier usiamo
la variante canonica: mediana e winsorizzazione). Così si legge a colpo d'occhio a quale rumore ciascun modello è più
sensibile. Lo facciamo per *accuracy* e *F1 (macro)*.

In [ ]:
for m in ['NN base', 'NN pro', 'RF']:
    plot_aggregate_per_model(m, AGG[m], 'accuracy')
    plot_aggregate_per_model(m, AGG[m], 'f1_macro')

## Tabella riassuntiva e grafici sintetici
Tabella baseline (0%) vs caso peggiore (40%) per modello / rumore / metrica, salvata anche in `risultati_summary.csv`.
Seguono: **heatmap** dell'accuracy nel caso peggiore, **bar chart** della degradazione (0%→40%) e **ranking** di
robustezza complessiva.

In [ ]:
def get_val(df, livello, metric):
    r = df[df['livello'] == livello]
    return float(r[metric].iloc[0]) if len(r) else float('nan')

summary_rows = []
for m in ['NN base', 'NN pro', 'RF']:
    for noise, df_ in AGG[m].items():
        for metric in ['accuracy', 'f1_macro', 'auc']:
            v0, v40 = get_val(df_, 0, metric), get_val(df_, 40, metric)
            summary_rows.append({'Modello': m, 'Rumore': noise, 'Metrica': metric,
                                 '@0%': f'{v0:.4f}', '@40%': f'{v40:.4f}', 'Delta': f'{v40 - v0:+.4f}'})

df_summary = pd.DataFrame(summary_rows)
print('=== TABELLA RIASSUNTIVA — Baseline (0%) vs Caso Peggiore (40%) ===')
print(df_summary.to_string(index=False))
df_summary.to_csv('risultati_summary.csv', index=False)
print('\nSalvata in risultati_summary.csv')

In [ ]:
noises = list(AGG['NN base'].keys())
models = ['NN base', 'NN pro', 'RF']

# 1) Heatmap accuracy @40% (modello x rumore)
mat = np.array([[get_val(AGG[m][n], 40, 'accuracy') for n in noises] for m in models])
plt.figure(figsize=(11, 4))
sns.heatmap(mat, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.5, vmax=1.0,
            xticklabels=noises, yticklabels=models)
plt.title('Accuracy nel caso peggiore (40%) — modello x rumore', fontweight='bold')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

# 2) Degradazione (acc@0 - acc@40)
x = np.arange(len(noises)); width = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(models):
    deg = [get_val(AGG[m][n], 0, 'accuracy') - get_val(AGG[m][n], 40, 'accuracy') for n in noises]
    ax.bar(x + i * width, deg, width, label=m, color=PALETTE[i])
ax.set_xticks(x + width); ax.set_xticklabels(noises, rotation=20, ha='right')
ax.set_ylabel('Calo di accuracy (0% -> 40%)')
ax.set_title('Robustezza: degradazione per tipo di rumore', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

# 3) Ranking robustezza complessiva (accuracy media @40% sui rumori)
rank = {m: float(np.mean([get_val(AGG[m][n], 40, 'accuracy') for n in noises])) for m in models}
plt.figure(figsize=(7, 4))
plt.bar(list(rank.keys()), list(rank.values()), color=PALETTE[:3])
plt.ylabel('Accuracy media @40%'); plt.ylim(0.5, 1.0)
plt.title('Ranking robustezza complessiva (media sui 6 rumori)', fontweight='bold')
plt.tight_layout(); plt.show()

## ROC: dati puliti vs 40% Feature Noise
Confronto delle curve ROC dei tre modelli, puliti vs forte feature noise (linee tratteggiate).

In [ ]:
X_fn40 = add_gaussian_noise(X_train_scaled, noise_level=0.40, seed=seed)
mb40, _ = train_base_network(X_fn40, y_train, X_val_scaled, y_val, random_state=seed)
mp40, _ = train_pro_network(X_fn40, y_train, X_val_scaled, y_val, random_state=seed)
rf40    = train_random_forest(X_fn40, y_train, random_state=seed)

pairs = [
    ('NN base pulito', m_base_clean, 'blue',  '-'),
    ('NN pro pulito',  m_pro_clean,  'green', '-'),
    ('RF pulito',      rf_clean,     'red',   '-'),
    ('NN base 40%FN',  mb40,         'blue',  '--'),
    ('NN pro 40%FN',   mp40,         'green', '--'),
    ('RF 40%FN',       rf40,         'red',   '--'),
]
plt.figure(figsize=(10, 7))
for name, model, c, ls in pairs:
    sc = model_proba(model, X_test_scaled)
    fpr, tpr, _ = roc_curve(y_test, sc)
    plt.plot(fpr, tpr, color=c, ls=ls, lw=2, label=f'{name} (AUC={roc_auc_score(y_test, sc):.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC — Dati puliti vs 40% Feature Noise', fontweight='bold')
plt.legend(loc='lower right', fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Intervalli di Confidenza — Riproducibilità
*"I risultati cambiano perché i dati sono sporchi o perché cambia il seed?"* Per rispondere, ripetiamo l'addestramento
con **N=7 seed** diversi e calcoliamo media e **intervallo di confidenza al 95%** (\(\bar{x} \pm 1.96\,\sigma/\sqrt{N}\)).
Limitiamo l'analisi (computazionalmente pesante) a **feature noise** e **label noise**, sui tre modelli.

- IC ampio → alta varianza dovuta al seed (rumore dell'addestramento).
- Media che cala sistematicamente → degrado reale dovuto ai dati sporchi.

In [ ]:
N_RUNS = 7
CI_SEEDS = [42, 7, 13, 99, 2024, 314, 17]
CI_LEVELS = [0.0] + NOISE_LEVELS


def ci_experiment(model_kind, noise_type, noise_level, seeds=CI_SEEDS):
    accs, aucs = [], []
    for s in seeds:
        Xtr, ytr = X_train_scaled, np.array(y_train)
        if noise_level > 0:
            if noise_type == 'fn':
                Xtr = add_gaussian_noise(X_train_scaled, noise_level=noise_level, seed=s)
            elif noise_type == 'ln':
                ytr = add_label_noise_random(y_train, noise_level=noise_level, seed=s)
        if model_kind == 'NN base':
            m, _ = train_base_network(Xtr, ytr, X_val_scaled, y_val, random_state=s)
        elif model_kind == 'NN pro':
            m, _ = train_pro_network(Xtr, ytr, X_val_scaled, y_val, random_state=s)
        else:
            m = train_random_forest(Xtr, ytr, random_state=s)
        proba = model_proba(m, X_test_scaled)
        y_pred = (proba > 0.5).astype(int)
        accs.append(accuracy_score(y_test, y_pred))
        aucs.append(roc_auc_score(y_test, proba))

    def ci(vals):
        a = np.array(vals); mn = a.mean(); sd = a.std(ddof=1)
        return mn, sd, 1.96 * sd / np.sqrt(len(a))

    am, asd, aci = ci(accs); um, usd, uci = ci(aucs)
    return {'acc_mean': am, 'acc_std': asd, 'acc_ci': aci,
            'auc_mean': um, 'auc_std': usd, 'auc_ci': uci}


ci_data = {}
for kind in ['NN base', 'NN pro', 'RF']:
    for nt, nice in [('fn', 'Feature noise'), ('ln', 'Label noise')]:
        print(f'Calcolo IC: {kind} / {nice} ...')
        ci_data[(kind, nt)] = {lvl: ci_experiment(kind, nt, lvl) for lvl in CI_LEVELS}
print('Intervalli di confidenza calcolati.')

In [ ]:
def plot_ci(ci_dict, metric='acc', label='', color='steelblue', ax=None):
    levels = sorted(ci_dict.keys())
    means = [ci_dict[l][f'{metric}_mean'] for l in levels]
    half  = [ci_dict[l][f'{metric}_ci']   for l in levels]
    xp = [l * 100 for l in levels]
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    ax.plot(xp, means, marker='o', lw=2, color=color, label=label)
    ax.fill_between(xp, [m - c for m, c in zip(means, half)],
                    [m + c for m, c in zip(means, half)], alpha=0.2, color=color)
    ax.set_xlabel('Livello di rumore (%)'); ax.set_ylabel(metric.upper()); ax.grid(alpha=0.3)
    return ax

cols = {'NN base': PALETTE[0], 'NN pro': PALETTE[1], 'RF': PALETTE[2]}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for kind in ['NN base', 'NN pro', 'RF']:
    plot_ci(ci_data[(kind, 'fn')], 'acc', kind, cols[kind], axes[0])
    plot_ci(ci_data[(kind, 'ln')], 'acc', kind, cols[kind], axes[1])
axes[0].set_title('Accuracy con IC 95% — Feature Noise', fontweight='bold'); axes[0].legend()
axes[1].set_title('Accuracy con IC 95% — Label Noise', fontweight='bold'); axes[1].legend()
plt.suptitle('Intervalli di Confidenza al 95% (7 seed distinti)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('\n=== TABELLA INTERVALLI DI CONFIDENZA (Accuracy) ===')
print(f'{"Modello":<10}{"Rumore":>14}{"Livello":>9}{"Media":>9}{"Std":>9}{"IC95":>10}')
print('-' * 61)
for (kind, nt), d in ci_data.items():
    nice = 'Feature' if nt == 'fn' else 'Label'
    for lvl in sorted(d):
        r = d[lvl]
        print(f'{kind:<10}{nice:>14}{int(lvl*100):>8}%{r["acc_mean"]:>9.4f}{r["acc_std"]:>9.4f}{r["acc_ci"]:>9.4f}')

# Modello Non Supervisionato (K-Means)
Il K-Means raggruppa senza usare le etichette: il **Label Noise** non lo influenza (usa solo le feature), mentre il
**Feature Noise** degrada la qualità dei cluster. Misuriamo ARI, AMI, NMI, purezza e silhouette.

In [ ]:
def cluster_quality(X_scaled, y_true, k=2, random_state=42):
    km = KMeans(n_clusters=k, n_init=10, random_state=random_state)
    labels = km.fit_predict(X_scaled)
    y_arr = np.array(y_true)
    purity = sum(np.bincount(y_arr[labels == c]).max() for c in np.unique(labels)) / len(y_arr)
    metrics = {
        'ARI':        adjusted_rand_score(y_arr, labels),
        'AMI':        adjusted_mutual_info_score(y_arr, labels),
        'NMI':        normalized_mutual_info_score(y_arr, labels),
        'purezza':    purity,
        'silhouette': silhouette_score(X_scaled, labels, sample_size=2000, random_state=random_state),
    }
    return metrics, labels, km

q_clean, labels_clean, _ = cluster_quality(X_train_scaled, y_train)
print('K-means (k=2) dati puliti:')
for m, v in q_clean.items():
    print(f'  {m:11s}: {v:.4f}')

In [ ]:
pc = PCA(n_components=2, random_state=seed).fit_transform(X_train_scaled)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].scatter(pc[:, 0], pc[:, 1], c=labels_clean, cmap='coolwarm', s=6, alpha=0.4)
ax[0].set_title('Cluster K-Means (k=2)')
ax[1].scatter(pc[:, 0], pc[:, 1], c=np.array(y_train), cmap='coolwarm', s=6, alpha=0.4)
ax[1].set_title('Classi reali (gamma / hadron)')
for a in ax:
    a.set_xlabel('PC1'); a.set_ylabel('PC2')
plt.suptitle('PCA — geometria dei cluster (dati puliti)', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
feature_levels = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]
print('--- Effetto del Feature Noise su K-Means ---')
print(f'{"sigma":>6}{"ARI":>8}{"AMI":>8}{"silhouette":>12}{"purezza":>10}')
for lvl in feature_levels:
    X_noisy = add_gaussian_noise(X_train_scaled, noise_level=lvl, seed=seed)
    q, _, _ = cluster_quality(X_noisy, y_train)
    print(f'{lvl:>6.2f}{q["ARI"]:>8.3f}{q["AMI"]:>8.3f}{q["silhouette"]:>12.3f}{q["purezza"]:>10.3f}')

y_train_flip = add_label_noise_random(y_train, noise_level=0.40, seed=seed)
_, labels_lab, _ = cluster_quality(X_train_scaled, y_train_flip)
print(f'\nARI(clustering pulito, clustering label-noise 40%) = {adjusted_rand_score(labels_clean, labels_lab):.3f}')
print('K-Means usa solo le feature -> il Label Noise non sposta i cluster (ARI=1.0).')

# Conclusioni
Sintesi attesa dallo studio (verificare con i numeri prodotti sopra):

- **Feature noise**: degrado graduale; la **NN Pro** (L2 + dropout + early stopping) e il **Random Forest** sono più
  stabili della NN Base, che tende ad adattarsi al rumore.
- **Label noise (random e borderline)**: il rumore sulle etichette è il più dannoso per le reti, che memorizzano gli
  errori (gap train/val crescente). Il **borderline** degrada più del random a parità di percentuale. Il RF resiste meglio.
- **Missing values**: con imputazione l'impatto è contenuto; **media e mediana** danno risultati simili (lieve vantaggio
  della mediana su feature asimmetriche).
- **Outlier**: lasciarli penalizza soprattutto le reti; la **winsorizzazione** recupera quasi del tutto le performance,
  l'imputazione è una buona alternativa.
- **Missing feature**: rimuovere le feature più importanti (es. `fAlpha`) causa il calo maggiore; i modelli ensemble
  ridistribuiscono meglio sull'informazione residua.
- **Intervalli di confidenza**: gli IC stretti su feature/label noise indicano che il **calo è un segnale reale**, non
  semplice varianza da seed.

**Idee per ulteriori grafici** (oltre a quelli già prodotti):
- *Overfit gap* (accuracy − val_accuracy finale) dalle history, per quantificare l'overfitting NN Base vs NN Pro.
- *Curve ROC differenziali* NN Pro − NN Base lungo l'FPR.
- *Recall/precision per classe* al variare del rumore (gamma vs hadron).
- *Larghezza degli IC* in funzione del livello, per separare varianza-da-seed e degrado-da-dati.
- *Heatmap aggregata su F1* (oltre a quella su accuracy) e *ranking* per F1.